In [1]:
import random
import numpy as np

In [2]:
class Neuron:
    def __init__(self, data,children = ()):
        self.data = data
        self.grad = 0
        self.children = children
        self._backward = lambda : None
    def __mul__(self,other):
        other = other if isinstance(other,Neuron) else Neuron(other)
        out = Neuron(self.data*other.data)
        out.children = (self,other)
        def _backward():
            self.grad += other.data*out.grad
            other.grad += self.data*out.grad
        out._backward = _backward
        return out
    def __add__(self,other):
        other = other if isinstance(other,Neuron) else Neuron(other)
        out = Neuron(self.data + other.data)
        out.children = (self,other)
        def _backward():
            self.grad += 1.0*out.grad
            other.grad += 1.0*out.grad
        out._backward = _backward
        return out
    def __sub__(self,other):
        other = other if isinstance(other,Neuron) else Neuron(other)
        out = Neuron(self.data - other.data)
        out.children = (self,other)
        def _backward():
            self.grad += 1.0*out.grad
            other.grad -= 1.0*out.grad
        out._backward = _backward
        return out
    def relu(self):
        out = Neuron(self.data) if self.data > 0 else Neuron(0)
        out.children = (self,)
        def _backward():
            self.grad += out.grad*1 if self.data > 0 else 0
        out._backward = _backward
        return out
    def __truediv__(self,other):
        other = other if isinstance(other,Neuron) else Neuron(other)
        try:
            out = self.data/other.data
            return Neuron(out)
        except ZeroDivisionError:
            out = Neuron(0)
        def _backward():
            self.grad += out.grad/other.data
            other.grad += -(self.data/((other.data)**2))*out.grad
        out.children = (self,other)
        out._backward = _backward
        return out
    def __pow__(self,other : float):
        # other = other if isinstance(other,Neuron) else Neuron(other) we are not doing this cuz
        # if other.data < 0 we might have to face a problem of get undefined
        # to avoid that I am considering other over here as an int variable only
        def _backward():
            self.grad += (other*(self.data**(other - 1)))*out.grad
        out = Neuron(self.data**other)
        out.children = (self,)
        out._backward = _backward
        return out
    def __radd__(self,other):
        other = other if isinstance(other,Neuron) else Neuron(other)
        out = Neuron(self.data + other.data)
        out.children = (self,other)
        def _backward():
            self.grad += 1.0*out.grad
            other.grad += 1.0*out.grad
        out._backward = _backward
        return out
    def __rsub__(self,other):
        other = other if isinstance(other,Neuron) else Neuron(other)
        out = (self.data - other.data)
        out.children = (self,other)
        def _backward():
            self.grad += 1.0*out.grad
            other.grad -= 1.0*out.grad
        out._backward = _backward
        return out
    def __rmul__(self,other):
        other = other if isinstance(other,Neuron) else Neuron(other)
        out = Neuron(self.data * other.data)
        out.children = (self,other)
        def _backward():
            self.grad += other.data*out.grad
            other.grad += self.data*out.grad
        out._backward = _backward
        return out
    def __rtruediv__(self,other):
        other = other if isinstance(other,Neuron) else Neuron(other)
        try:
            out = self.data/other.data
            return Neuron(out)
        except ZeroDivisionError:
            out = Neuron(0)
        def _backward():
            self.grad += out.grad/other.data
            other.grad += -(self.data/((other.data)**2))*out.grad
        out.children = (self,other)
        out._backward = _backward
        return out
    def __rpow__(self,other : float):
        out = Neuron(self.data**other)
        def _backward():
            self.grad += (other*(self.data**(other - 1)))*out.grad
        out.children = (self,other)
        out._backward = _backward
        return out
    def __neg__(self):
        out = Neuron(-self.data)
        out.children = (self,)
        def _backward():
            self.grad += -1.0*out.grad
        out._backward = _backward
        return out
    def __repr__(self):
        return f"Neuron({self.data} grad {self.grad})"
    def backward(self):
        self.grad = 1
        visited = set()
        topo = list()
        def topo_sort(n : Neuron,visited : set,topo : list):
            if n not in visited:
                visited.add(n)
            for child in n.children:
                topo_sort(child, visited, topo)
            topo.append(n)
        topo_sort(self,visited, topo)
        for n in reversed(topo):
            n._backward()
class Module:

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0

    def parameters(self):
        return []

class Node(Module):

    def __init__(self, nin, nonlin=True):
        self.w = [Neuron(random.uniform(-np.sqrt(6/nin),np.sqrt(6/nin))) for _ in range(nin)]
        self.b = Neuron(0)
        self.nonlin = nonlin

    def __call__(self, x):
        act = sum((wi*xi for wi,xi in zip(self.w, x)), self.b)
        return act.relu() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]

    def __repr__(self):
        return f"{'ReLU' if self.nonlin else 'Linear'}Neuron({len(self.w)})"

class Layer(Module):

    def __init__(self, nin, nout, **kwargs):
        self.neurons = [Node(nin, **kwargs) for _ in range(nout)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

    def __repr__(self):
        return f"Layer of [{', '.join(str(n) for n in self.neurons)}]"

class MLP(Module):

    def __init__(self, nin, nouts):
        self.x = None
        self.y = None
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1], nonlin=i!=len(nouts)-1) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def __repr__(self):
        return f"MLP of [{', '.join(str(layer) for layer in self.layers)}]"
    def fit(self,x,y,optimizer = 'adam',epochs = 1000):
        learning_rate = 0.0001
        self.x = x
        self.y = y
        self.optimizer = optimizer
        y_pred = [self(xs) for xs in self.x]
        cost = sum((yout - ygt) ** 2 for ygt, yout in zip(self.y, y_pred))
        def adam():
            m = [0]*len(self.parameters())
            v = [0]*len(self.parameters())
            m =[0.9*w1 + 0.1*grad for w1,grad in zip(m, [p.grad for p in self.parameters()])]
            v =[0.999*v1 + 0.001*(grad**2) for v1,grad in zip(v, [p.grad for p in self.parameters()])]
            hat_m = [(_m)/(1 - (0.9)**(_ + 1)) for _m in m]
            hat_v = [(_v)/(1 - (0.999)**(_ + 1)) for _v in v]
            r_v = [np.sqrt(1/(_hat_v + 0.1**8)) for _hat_v in hat_v]
            for params,_hat_m,_r_v in zip(self.parameters(),hat_m,r_v):
                params.data -= learning_rate*_hat_m*_r_v
        def RMSprop():
            v = [0]*len(self.parameters())
            hat_v = [(_v)/(1 - (0.999)**(_ + 1)) for _v in v]
            r_v = [np.sqrt(1/(_hat_v + 0.1**8)) for _hat_v in hat_v]
            for params,_hat_m,_r_v in zip(self.parameters(),r_v):
                params.data -= learning_rate*_r_v
        def momentum():
            m = [0]*len(self.parameters())
            hat_m = [(_m)/(1 - (0.9)**(_ + 1)) for _m in m]
            for params,_hat_m,_r_v in zip(self.parameters(),hat_m):
                params.data -= learning_rate*_hat_m
        if self.optimizer == 'adam':
            update_step = adam
        elif self.optimizer =='RMSprop':
            update_step = RMSprop
        elif self.optimizer =='momentum':
            update_step = momentum
        else:
            raise ValueError(f'Unknown Optimizer : {self.optimizer}')
        for _ in range(epochs):
            self.zero_grad()
            cost.backward()
            update_step()
            y_pred = [self(xs) for xs in self.x]
            cost = sum((yout - ygt) ** 2 for ygt, yout in zip(self.y, y_pred))
            if _%50 == 0:
                print(f"Epoch {_} | Loss: {cost.data:.4f}")











In [3]:
x = Neuron(3.0)
a = x * 2 + x * 5 + x**2  # 'a' remembers how to update 'x'
b = a.relu()   # Does this delete 'a's memory?
b.backward()

print(f"x.grad should be 2. It is: {x.grad}")


x.grad should be 2. It is: 13.0


In [9]:
# 1. Create the dataset (XOR Problem)
# Inputs: (0,0), (0,1), (1,0), (1,1)
xs = [
    [2.0, 3.0],
    [3.0, -1.0],
    [-0.5, 0.5],
    [1.0, 1.0]
]
# Targets (Labels): 1 if inputs are similar, -1 if different
ys = [1.0, -1.0, -1.0, 1.0]

# 2. Initialize the Network
# 2 inputs -> 3 hidden neurons -> 1 output neuron
n = MLP(2, [3, 1])
n.fit(xs, ys,epochs = 4000)

# 3. Training Loop
# learning_rate = 0.05
#
# for k in range(500): # Run for 500 epochs
#
#     # --- Forward Pass ---
#     ypred = [n(x) for x in xs]
#
#     # --- Calculate Loss (Mean Squared Error) ---
#     loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
#
#     # --- Backward Pass ---
#     n.zero_grad() # Reset old gradients to 0
#     loss.backward()  # Calculate new gradients
#
#     # --- Update (Gradient Descent) ---
#     for p in n.parameters():
#         p.data += -learning_rate * p.grad
#
#     # Print progress
#     if k % 50 == 0:
#         print(f"Epoch {k} | Loss: {loss.data:.4f}")

# 4. Final Predictions
print("\nFinal Predictions:")
for x, y in zip(xs, ys):
    pred = n(x)
    print(f"Input: {x} | Target: {y} | Prediction: {pred.data:.4f}")

Epoch 0 | Loss: 26.4671
Epoch 50 | Loss: 26.0576
Epoch 100 | Loss: 25.4592
Epoch 150 | Loss: 24.7138
Epoch 200 | Loss: 23.8664
Epoch 250 | Loss: 22.9467
Epoch 300 | Loss: 21.9769
Epoch 350 | Loss: 20.9746
Epoch 400 | Loss: 19.9544
Epoch 450 | Loss: 18.9284
Epoch 500 | Loss: 17.9070
Epoch 550 | Loss: 16.8988
Epoch 600 | Loss: 15.9114
Epoch 650 | Loss: 14.9512
Epoch 700 | Loss: 14.0236
Epoch 750 | Loss: 13.1331
Epoch 800 | Loss: 12.2835
Epoch 850 | Loss: 11.4779
Epoch 900 | Loss: 10.7188
Epoch 950 | Loss: 10.0081
Epoch 1000 | Loss: 9.3471
Epoch 1050 | Loss: 8.7370
Epoch 1100 | Loss: 8.1729
Epoch 1150 | Loss: 7.6405
Epoch 1200 | Loss: 7.1391
Epoch 1250 | Loss: 6.6687
Epoch 1300 | Loss: 6.2292
Epoch 1350 | Loss: 5.8201
Epoch 1400 | Loss: 5.4426
Epoch 1450 | Loss: 5.1282
Epoch 1500 | Loss: 4.8314
Epoch 1550 | Loss: 4.5519
Epoch 1600 | Loss: 4.2896
Epoch 1650 | Loss: 4.0439
Epoch 1700 | Loss: 3.8145
Epoch 1750 | Loss: 3.6010
Epoch 1800 | Loss: 3.4027
Epoch 1850 | Loss: 3.2119
Epoch 1900 | Lo